# Compute experiment data processing

This notebook:
- Creates a subdirectory in `analysis_plots` for each `compute_*.csv` in `measurements`, wipes and recreates them each run
- Copies each CSV into its subdirectory and adds derived columns
- **e** columns (e-prefixed): bound between -1 and +1
- **i** columns (i-prefixed): bound by ±5×IREFP (e.g. 500nA when IREFP=100nA)
- Plots and filters will be added back later.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import shutil

def round_to_n_sig_figs(x, n=5):
    """Round scalar to n significant figures. Keeps 0, NaN, Inf unchanged."""
    if pd.isna(x) or x == 0 or np.isinf(x):
        return x
    try:
        return round(x, n - 1 - int(np.floor(np.log10(abs(x)))))
    except (ValueError, OverflowError):
        return x

def df_round_sig_figs(df, n=5):
    """Round all float columns to n significant figures (avoids 9.99...e-09 etc.)."""
    out = df.copy()
    for col in out.columns:
        if out[col].dtype.kind == 'f':
            out[col] = out[col].apply(lambda x: round_to_n_sig_figs(x, n))
    return out

# Absolute paths: same locations regardless of cwd (only .ipynb files stay in analysis_plots)
cwd = Path(os.getcwd()).resolve()
if cwd.name == 'analysis_plots':
    analysis_plots_dir = cwd
    measurements_dir = cwd.parent / 'measurements'
else:
    analysis_plots_dir = cwd / 'analysis_plots'
    measurements_dir = cwd / 'measurements'
analysis_plots_dir = analysis_plots_dir.resolve()
measurements_dir = measurements_dir.resolve()

print(f"analysis_plots_dir: {analysis_plots_dir}")
print(f"measurements_dir: {measurements_dir}")

# Step 1: Remove ALL compute_* subdirectories in analysis_plots (only notebooks remain)
if analysis_plots_dir.exists():
    for item in list(analysis_plots_dir.iterdir()):
        if item.is_dir() and item.name.startswith('compute_'):
            try:
                shutil.rmtree(item)
                print(f"Removed: {item.name}")
            except OSError as e:
                print(f"Warning: could not remove {item.name}: {e}")

# Step 2: Find all compute_*.csv in measurements
compute_csvs = sorted(measurements_dir.glob('compute_*.csv'))
print(f"Found {len(compute_csvs)} compute CSV(s): {[f.name for f in compute_csvs]}")

# Columns: value / IREFP, 3 decimal places
divide_by_irefp = ['KGAIN1', 'KGAIN2', 'TRIM1', 'TRIM2']
# Columns: A*2/IREFP - 1
linear_irefp = ['X1', 'X2', 'F11', 'F12', 'IMEAS']

# Voltage measurement columns (measured at each current source terminal)
voltage_meas_cols = ['vIMEAS', 'vX1', 'vX2', 'vTRIM1', 'vTRIM2', 'vF11', 'vF12',
                     'vKGAIN1', 'vKGAIN2', 'vIREFP']

processed_paths = []
for csv_path in compute_csvs:
    csv_path = csv_path.resolve()
    subdir_name = csv_path.stem  # e.g. compute_20260115_161903
    subdir = analysis_plots_dir / subdir_name
    subdir.mkdir(parents=True, exist_ok=True)
    dest_csv = subdir / csv_path.name
    shutil.copy2(str(csv_path), str(dest_csv))
    df = pd.read_csv(str(dest_csv))

    # Rename OUT1_current -> IOUT_meas
    if 'OUT1_current' in df.columns:
        df.rename(columns={'OUT1_current': 'IOUT_meas'}, inplace=True)

    if 'IREFP' not in df.columns:
        print(f"  Skip {csv_path.name}: no IREFP column")
        continue
    irefp = df['IREFP'].replace(0, np.nan)

    for col in divide_by_irefp:
        if col in df.columns:
            df['e' + col] = (df[col] / irefp).round(3)

    for col in linear_irefp:
        if col in df.columns:
            df['e' + col] = df[col] * 2 / irefp - 1

    # Clip all e* columns to [-1, 1]
    e_cols = [c for c in df.columns if c.startswith('e')]
    for c in e_cols:
        df[c] = df[c].clip(-1.0, 1.0)

    # New formulas (row-wise)
    df['eXhat'] = df['eX1'] * df['eF11'] + df['eX2'] * df['eF12']
    df['eY'] = df['eIMEAS'] - df['eXhat']
    df['eXhatplus'] = df['eY'] * df['eKGAIN1'] + df['eXhat']
    df['iXhatplus'] = (df['eXhatplus'] + 1) * df['IREFP'] / 2

    # iDeltaX: if PPG_state is ERASE then iXhatplus - X1, else X1 - iXhatplus; minimum 1e-10
    is_erase = (df['PPG_state'] == 'ERASE')
    df['iDeltaX'] = np.where(is_erase, df['iXhatplus'] - df['X1'], df['X1'] - df['iXhatplus'])
    df['iDeltaX'] = df['iDeltaX'].clip(lower=1e-10)

    df['IOUT_sim'] = df['X1'] * df['TRIM1'] / df['iDeltaX']

    # Clip all e* columns to [-1, 1]
    e_cols = [c for c in df.columns if c.startswith('e')]
    for c in e_cols:
        df[c] = df[c].clip(-1.0, 1.0)

    # Clip all i* columns to ±5*IREFP (per row, e.g. 500nA when IREFP=100nA)
    i_cols = [c for c in df.columns if c.startswith('i')]
    irefp_5 = 5.0 * df['IREFP']
    for c in i_cols:
        df[c] = df[c].clip(-irefp_5, irefp_5)

    # Reorder columns: IOUT_meas next to IOUT_sim, then v columns at end
    cols = list(df.columns)
    if 'IOUT_meas' in cols and 'IOUT_sim' in cols:
        cols.remove('IOUT_meas')
        sim_idx = cols.index('IOUT_sim')
        cols.insert(sim_idx, 'IOUT_meas')

    # Move each voltage column right after its corresponding current column
    # e.g. vX1 → right after X1, vF11 → right after F11, etc.
    for vc in voltage_meas_cols:
        if vc not in cols:
            continue
        base = vc[1:]  # strip leading 'v'
        cols.remove(vc)
        if base in cols:
            cols.insert(cols.index(base) + 1, vc)
        else:
            cols.insert(0, vc)  # fallback: put at front
    df = df[cols]

    # Round all floats to 5 significant figures (fixes 9.999...e-09 -> 1e-8 etc.)
    df = df_round_sig_figs(df, n=5)

    df.to_csv(str(dest_csv), index=False)
    processed_paths.append(dest_csv)

    found_v = [vc for vc in voltage_meas_cols if vc in df.columns]
    print(f"  Created {subdir_name}/ and saved modified {csv_path.name}"
          + (f"  (voltage cols: {found_v})" if found_v else "  (no voltage cols in source)"))

print(f"\nDone. Processed {len(processed_paths)} CSV(s).")
print("Derived columns: eKGAIN1, eKGAIN2, eTRIM1, eTRIM2, eX1, eX2, eF11, eF12, eIMEAS, eXhat, eY, eXhatplus, iXhatplus, iDeltaX, IOUT_sim")
print(f"Voltage measurement columns (when present): {voltage_meas_cols}")

analysis_plots_dir: C:\Users\chamberlain\Documents\kalman_lab\analysis_plots
measurements_dir: C:\Users\chamberlain\Documents\kalman_lab\measurements
Removed: compute_20260115_161903
Removed: compute_20260115_171219
Removed: compute_20260115_174521
Removed: compute_20260301_124206
Removed: compute_20260301_135621
Removed: compute_20260301_143107
Removed: compute_20260301_150543
Found 7 compute CSV(s): ['compute_20260115_161903.csv', 'compute_20260115_171219.csv', 'compute_20260115_174521.csv', 'compute_20260301_124206.csv', 'compute_20260301_135621.csv', 'compute_20260301_143107.csv', 'compute_20260301_150543.csv']
  Created compute_20260115_161903/ and saved modified compute_20260115_161903.csv  (no voltage cols in source)
  Created compute_20260115_171219/ and saved modified compute_20260115_171219.csv  (no voltage cols in source)
  Created compute_20260115_174521/ and saved modified compute_20260115_174521.csv  (no voltage cols in source)
  Created compute_20260301_124206/ and saved

In [2]:
# (Data filters removed — add back later)

In [3]:
# (Plot config removed — add back later)

In [4]:
# (Plot function removed — add back later)

In [5]:
# (Plot generation removed — add back later)

In [6]:
# IOUT report: counts by range for both model (IOUT_sim) and meas (IOUT_meas)
# All processed datasets summarized in a single file.
# Categories: <1nA, 1nA–100nA, 100nA–1µA, >1µA

NA_1 = 1e-9      # 1 nA
NA_100 = 100e-9  # 100 nA
UA_1 = 1e-6      # 1 µA

def make_report_row(dataset_name, series, source_type, total):
    """Build one report row for a given current series."""
    below_1na = (series < NA_1).sum()
    within_1na_100na = ((series >= NA_1) & (series <= NA_100)).sum()
    above_100na = ((series > NA_100) & (series <= UA_1)).sum()
    above_1ua = (series > UA_1).sum()

    def pct(x, n):
        return round_to_n_sig_figs(100 * x / n, 3) if n else 0

    return {
        'dataset': dataset_name,
        'type': source_type,
        'total': total,
        'below_1nA': below_1na,
        'within_1nA_to_100nA': within_1na_100na,
        'above_100nA': above_100na,
        'above_1uA': above_1ua,
        'pct_below_1nA': pct(below_1na, total),
        'pct_within_1nA_100nA': pct(within_1na_100na, total),
        'pct_above_100nA': pct(above_100na, total),
    }

report_rows = []
for path in processed_paths:
    df = pd.read_csv(str(path))
    dataset_name = path.parent.name
    total = len(df)

    if 'IOUT_sim' in df.columns:
        report_rows.append(make_report_row(dataset_name, df['IOUT_sim'].astype(float), 'model', total))
    if 'IOUT_meas' in df.columns:
        report_rows.append(make_report_row(dataset_name, df['IOUT_meas'].astype(float), 'meas', total))

report_df = pd.DataFrame(report_rows)
report_path = analysis_plots_dir / 'iOut1_measurement_report.csv'
report_df.to_csv(str(report_path), index=False)
print(f"Report saved: {report_path}")
print(report_df.to_string(index=False))

Report saved: C:\Users\chamberlain\Documents\kalman_lab\analysis_plots\iOut1_measurement_report.csv
                dataset  type  total  below_1nA  within_1nA_to_100nA  above_100nA  above_1uA  pct_below_1nA  pct_within_1nA_100nA  pct_above_100nA
compute_20260115_161903 model    756         54                   78          196        428           7.14                10.300          25.9000
compute_20260115_161903  meas    756         31                   89          320        316           4.10                11.800          42.3000
compute_20260115_171219 model    756         54                   78          196        428           7.14                10.300          25.9000
compute_20260115_171219  meas    756         29                   43          168        516           3.84                 5.690          22.2000
compute_20260115_174521 model    756         54                   78          196        428           7.14                10.300          25.9000
compute_20260115_1